### Set up

In [11]:
import pandas as pd
import requests
import json
import dotenv
import os, time, requests
dotenv.load_dotenv()

True

In [3]:
key = os.getenv('key')
token = os.getenv("token")

In [ ]:
#user agent

botname = 'ds6021'
version = '0.0'
website = 'Class project'
email = 'qny2gg@virginia.edu'

useragent = f'{botname}/{version} ({website}; {email}) python-requests/{requests.__version__}'
useragent

'ds6001/0.0 (Class project; qny2gg@virginia.edu) python-requests/2.34.2'

In [ ]:
BASE = "https://api.themoviedb.org/3"

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {token}",
    "accept": "application/json",
    "User-Agent": useragent,
})

def get(path, **params):
    for attempt in range(5):
        r = session.get(f"{BASE}{path}", params=params, timeout=20)
        if r.status_code == 429:               
            time.sleep(int(r.headers.get("Retry-After", 2)))
            continue
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()
    return None

### Collect movie IDs by year

Loops through each release year from 1990 to 2025 collecting the IDs of movies that have at least 100 votes, most-voted first. It stores those IDs in a deduplicated list

In [12]:
def collect_ids(years, min_votes=100, pages_per_year=25):
    ids = {}
    for yr in years:
        for page in range(1, pages_per_year + 1):
            data = get("/discover/movie",
                       primary_release_year=yr,
                       **{"vote_count.gte": min_votes},
                       sort_by="vote_count.desc",
                       language="en-US",
                       page=page)
            if not data or not data["results"]:
                break
            for m in data["results"]:
                ids[m["id"]] = yr
            if page >= data["total_pages"]:
                break
            time.sleep(0.05)
        print(yr, len(ids))
    return list(ids)

movie_ids = collect_ids(range(1990, 2026))
len(movie_ids)

1990 201
1991 404
1992 604
1993 826
1994 1064
1995 1301
1996 1545
1997 1791
1998 2070
1999 2368
2000 2659
2001 2972
2002 3319
2003 3693
2004 4101
2005 4517
2006 5017
2007 5517
2008 6017
2009 6517
2010 7017
2011 7517
2012 8017
2013 8517
2014 9017
2015 9517
2016 10017
2017 10517
2018 11017
2019 11517
2020 12017
2021 12517
2022 13017
2023 13517
2024 14017
2025 14492


14492

### Pull details of the movie

Takes each movie ID from above andpulls that movie’s full details, then saves everything as a table

In [ ]:
def parse_movie(d):
    crew = d.get("credits", {}).get("crew", [])
    cast = d.get("credits", {}).get("cast", [])
    directors = [c["name"] for c in crew if c["job"] == "Director"]
    return {
        "tmdb_id": d["id"],
        "imdb_id": d.get("imdb_id"),
        "title": d["title"],
        "release_date": d.get("release_date"),
        "budget": d.get("budget"),
        "revenue": d.get("revenue"),
        "runtime": d.get("runtime"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])),
        "original_language": d.get("original_language"),
        "popularity": d.get("popularity"),
        "production_companies": "|".join(c["name"] for c in d.get("production_companies", [])),
        "director": "|".join(directors),
        "top_cast": "|".join(c["name"] for c in cast[:5]),
        "vote_count": d.get("vote_count"),
        "vote_average": d.get("vote_average"),   #this is our target var
    }

OUTFILE = "tmdb_movies.csv"

# Resume from a previous run if the file exists
if os.path.exists(OUTFILE):
    existing = pd.read_csv(OUTFILE)
    rows = existing.to_dict("records")
    done = set(existing["tmdb_id"])
    print(f"Resuming with {len(done)} movies already saved")
else:
    rows, done = [], set()

for i, mid in enumerate(movie_ids, 1):
    if mid in done:
        continue
    d = get(f"/movie/{mid}", append_to_response="credits", language="en-US")
    if d:
        rows.append(parse_movie(d))
    done.add(mid)

    if i % 500 == 0: 
        pd.DataFrame(rows).to_csv(OUTFILE, index=False)
        print(i, "of", len(movie_ids), "processed")
    time.sleep(0.03)


500 of 14492 processed
1000 of 14492 processed
1500 of 14492 processed
2000 of 14492 processed
2500 of 14492 processed
3000 of 14492 processed
3500 of 14492 processed
4000 of 14492 processed
4500 of 14492 processed
5000 of 14492 processed
5500 of 14492 processed
6000 of 14492 processed
6500 of 14492 processed
7000 of 14492 processed
7500 of 14492 processed
8000 of 14492 processed
8500 of 14492 processed
9000 of 14492 processed
9500 of 14492 processed
10000 of 14492 processed
10500 of 14492 processed
11000 of 14492 processed
11500 of 14492 processed
12000 of 14492 processed
12500 of 14492 processed
13000 of 14492 processed
13500 of 14492 processed
14000 of 14492 processed


NameError: name 'df' is not defined